Part 3:

In [ ]:
import pandas as pd
netid = 'smaciasa'
summary = pd.read_csv(f'{netid}_project_summary.csv', sep=';')
stats = pd.read_csv(f'{netid}_project_stats.csv', sep=';')
summary['Month'] = pd.to_datetime(summary['time'], unit='s').dt.strftime('%Y-%m')
summary['date'] = pd.to_datetime(summary['time'], unit='s').dt.strftime('%Y-%m-%d')
summary = summary.sort_values(['project_wocid', 'time'])
stats[['Project', 'LongestGapStart', 'LongestGapEnd', 'LongestGapLength']]

,Project,LongestGapStart,LongestGapEnd,LongestGapLength
0,antonyuryev_elsevierapi,2024-05,2024-07,3
1,bc_retina,2021-03,2024-12,46
2,caseywdunn_siphonophore_phylogeny_2017,2019-08,2023-01,42
3,giaf_hpipm,2022-12,2023-04,5
4,iic-jku_ddsim,2020-05,2020-05,1
5,malbecc_lio,2011-10,2013-06,21
6,neurodatawithoutborders_matnwb,2017-12,2018-01,2
7,prjemian_spec2nexus,2024-05,2025-04,12
8,spirit-code_spirit,2023-01,2023-01,1
9,stanfordnlp_glove,2023-10,2024-10,13


## Step 3.1


In [ ]:
cols = ['project_wocid', 'commit_sha1', 'author', 'time', 'commit message']
parts = []
for _, r in stats.iterrows():
    g = summary[summary['project_wocid'] == r['Project']]
    pre = g[g['Month'] < r['LongestGapStart']].tail(10)[cols].copy()
    post = g[g['Month'] > r['LongestGapEnd']].head(10)[cols].copy()
    pre.insert(0, 'pre/post', 'pre'); post.insert(0, 'pre/post', 'post')
    parts += [pre, post]
    print(f"{r['Project']}: {len(pre)} pre, {len(post)} post")
gap_commits = pd.concat(parts, ignore_index=True)
gap_commits.to_csv(f'{netid}_project_gap_commits.csv', sep=';', index=False)
print(gap_commits.shape)

antonyuryev_elsevierapi: 10 pre, 10 post
bc_retina: 10 pre, 5 post
caseywdunn_siphonophore_phylogeny_2017: 10 pre, 2 post
giaf_hpipm: 10 pre, 10 post
iic-jku_ddsim: 10 pre, 10 post
malbecc_lio: 10 pre, 10 post
neurodatawithoutborders_matnwb: 10 pre, 10 post
prjemian_spec2nexus: 10 pre, 10 post
spirit-code_spirit: 10 pre, 10 post
stanfordnlp_glove: 10 pre, 10 post
(187, 6)


In [ ]:
# reads the messages around each of the gaps
def show(df):
    for _, c in df.iterrows():
        print(f"   {pd.to_datetime(c['time'], unit='s'):%Y-%m-%d} | {c['author'].split('<')[0].strip()[:25]:25} | {str(c['commit message'])[:100]}")
for p in stats['Project']:
    print(f'\n=== {p}')
    for side in ['pre', 'post']:
        print(' --', side)
        show(gap_commits[(gap_commits['project_wocid'] == p) & (gap_commits['pre/post'] == side)])


=== antonyuryev_elsevierapi
 -- pre
   2024-01-16 | Anton Yuryev              | Add files via upload
   2024-01-16 | Anton Yuryev              | Add files via upload
   2024-04-15 | Anton Yuryev              | Add files via upload
   2024-04-15 | Anton Yuryev              | Add files via upload
   2024-04-15 | Anton Yuryev              | Add files via upload
   2024-04-15 | Anton Yuryev              | Add files via upload
   2024-04-15 | Anton Yuryev              | Add files via upload
   2024-04-23 | Anton Yuryev              | Read from RNEF file example added
   2024-04-23 | Anton Yuryev              | Add files via upload
   2024-04-23 | Anton Yuryev              | Add files via upload
 -- post
   2024-08-27 | Anton Yuryev              | Add files via upload
   2024-08-27 | Anton Yuryev              | Add files via upload
   2024-08-27 | Anton Yuryev              | Add files via upload
   2024-08-27 | Anton Yuryev              | Add files via upload
   2024-08-27 | Anton Yuryev   

## Who committed before and after the gap

In [ ]:
name = lambda a: a.split('<')[0].strip()
for _, r in stats.iterrows():
    g = summary[summary['project_wocid'] == r['Project']]
    before = set(g[g['Month'] < r['LongestGapStart']]['author'].map(name))
    after = g[g['Month'] > r['LongestGapEnd']]['author'].map(name)
    print(f"{r['Project']}: {after.nunique()} authors after the gap, {len(set(after) - before)} of them new")
    print('   top after:', after.value_counts().head(4).to_dict())

antonyuryev_elsevierapi: 1 authors after the gap, 0 of them new
   top after: {'Anton Yuryev': 118}
bc_retina: 1 authors after the gap, 0 of them new
   top after: {'Brian Cohn': 5}
caseywdunn_siphonophore_phylogeny_2017: 1 authors after the gap, 0 of them new
   top after: {'Casey Dunn': 2}
giaf_hpipm: 15 authors after the gap, 10 of them new
   top after: {'giaf': 134, 'sandmaennchen': 33, 'Gianluca Frison': 12, 'Jonathan Frey': 12}
iic-jku_ddsim: 33 authors after the gap, 31 of them new
   top after: {'Stefan Hillmich': 517, 'burgholzer': 445, 'renovate[bot]': 397, 'Lukas Burgholzer': 336}
malbecc_lio: 42 authors after the gap, 41 of them new
   top after: {'fedepedron': 951, 'gonzalodm': 246, 'Juan Pablo Darago': 225, 'Francisco': 224}
neurodatawithoutborders_matnwb: 30 authors after the gap, 28 of them new
   top after: {'ehennestad': 1635, 'Lawrence': 825, 'Lawrence Niu': 316, 'cechava': 249}
prjemian_spec2nexus: 3 authors after the gap, 0 of them new
   top after: {'Pete R Jemia

## Most recent 10 commits
WoC data ends around Nov 2025, so these are the last 10 commits WoC has for each project.

In [ ]:
for p in stats['Project']:
    print(f'\n=== {p}')
    show(summary[summary['project_wocid'] == p].tail(10))


=== antonyuryev_elsevierapi
   2025-10-09 | Anton Yuryev              | Add files via upload
   2025-10-09 | Anton Yuryev              | Add files via upload
   2025-10-09 | Anton Yuryev              | Add files via upload
   2025-10-09 | Anton Yuryev              | Add files via upload
   2025-10-09 | Anton Yuryev              | Add files via upload
   2025-10-24 | Anton Yuryev              | PostgresImporter optimized
   2025-10-24 | Anton Yuryev              | Update and rename readresnet.py to PostgresImport/readresnet.py lxml added to read <resnet> in secti
   2025-10-25 | Anton Yuryev              | Add files via upload
   2025-10-25 | Anton Yuryev              | Add files via upload
   2025-10-25 | Anton Yuryev              | Multithreading added

=== bc_retina
   2021-02-03 | bc                        | begin juan diagnostics
   2021-02-03 | bc                        | closing out juan-changes
   2021-02-17 | bc                        | op
   2021-02-17 | bc                   

   2025-08-11 | Nano                      | 🔧 REFACTORIZACIÓN MAYOR: Interfaz Python-LIO usa rutinas estándar ✨ CAMBIOS PRINCIPALES: - python_li
   2025-08-11 | Nano                      | Agregado un pyliosolo que debería aceptar los inputs de liosolo (y que se ejecuta igual)
   2025-08-21 | diegoarmino               | 1) g2g/Makefile.cuda: Disabled deprecated (as of cuda 11) cuda architecture codes. 2) lioamber/basis
   2025-09-05 | Jonathan Semelak          | add harris_data module
   2025-09-05 | Jonathan Semelak          | add ugly block to print DF data
   2025-09-05 | Jonathan Semelak          | modularize df verbosity
   2025-09-05 | Jonathan Semelak          | add Kin E computing and printint
   2025-09-05 | Jonathan Semelak          | remove unnecesary E1s calc
   2025-09-05 | Jonathan Semelak          | add df_verbosity keyword
   2025-09-10 | Jonathan Semelak          | unify mods for ML

=== neurodatawithoutborders_matnwb
   2025-10-27 | ehennestad                | Update 

## 3.1 / 3.2 Themes and gaps
Themes come from the assignment's list. "Other" covers CI/build config, tests, merges, data files and web uploads.
Current status rule: last commit before 2025-04-01 means Inactive (as of 2025-09-30). For spirit and lio the GitHub master branch is older but WoC has 2025 commits on other branches, so they count as Active. ElsevierAPI's repo was deleted after Oct 2025, so its status comes from WoC.

In [ ]:
# My judgments for Part 3 which are based on reading the commit messages in
# smaciasa_project_gap_commits.csv and the authors before and after each gap,
# and the GitHub pages (README, issues) of each project.

analysis = {
 'antonyuryev_elsevierapi': dict(
    BeforeThemes='Other:Feature development',
    AfterThemes='Other:Feature development',
    RecentThemes='Other:Feature development',
    HypothesizedGapReason='Short 3 month break by the only real developer. Anton Yuryev made 590 of the 615 commits before the gap and kept uploading the same kind of files (mostly Add files via upload) right before and right after it. Nothing in the messages points to a problem so it looks like he was busy with other work.',
    HasRecovered='Yes',
    WhyRecovered='The same developer came back in Aug 2024 and kept adding features (RepurposeDrug toxicity support and getTargets in Sep 2024 and a PostgreSQL importer with multithreading in Oct 2025).',
    WhoRecovered='Same person. All 118 commits after the gap are by Anton Yuryev and no new people joined.',
    Currentstatus='Active',
    Notes='A one person project that was very busy in 2021 and then slowed down but never really stopped. The 3 month gap in mid 2024 was easy to read because the same person picked up right where he left off. It was still active in Oct 2025 but the GitHub repo has since been deleted so stars and issues could not be checked.',
    reflection='''## Inactivity pattern
This project is basically one person. Anton Yuryev made almost every commit. It was really busy in 2021 (324 commits) and then slowed down each year. It only has one gap of 3 months or more which is May to July 2024.

## Was the gap easy or hard to read?
Easy. The 10 commits before and after the gap look the same. Most of them are "Add files via upload" which means he was uploading files through the GitHub website. Right before the gap he added an RNEF file reader example and right after it he added RepurposeDrug toxicity support and a getTargets function. Nothing in the messages talks about stopping or having a problem.

## Likely reason for the gap and recovery
My best guess is he was just busy with other work for a few months. He came back on his own in August 2024 and kept going into October 2025 with things like a PostgreSQL importer and multithreading. No new people joined.

## Extra note
The GitHub repo for this project has been deleted since then. So I could not look at issues or pull requests and all the evidence comes from the commit messages in WoC.
'''),

 'bc_retina': dict(
    BeforeThemes='Other:Feature development',
    AfterThemes='Other:Documentation updates',
    RecentThemes='Other:Documentation updates',
    HypothesizedGapReason='The author moved on. The tool came out of a 2015 Journal of Vision paper and most work was done by 2018. In the last commits before the gap (Feb 2021) Brian Cohn was committing from a fibronostics.com work email instead of his USC email which suggests he had a new job outside the lab. The last messages (closing out juan-changes and tutorial test) sound like wrapping up. Issue #53 (Jan 2021) was opened right before the gap and is still open.',
    HasRecovered='No',
    WhyRecovered='It did not really recover. The 5 commits in Jan 2025 were all made on one day and only touch CI files and a repomix.yml config and remove tutorial.md. That is cleanup not new development and nothing came after.',
    WhoRecovered='Same person (Brian Cohn). No new contributors.',
    Currentstatus='Inactive',
    Notes='Most of the work happened in 2014 to 2018 around a published paper and then it slowed down. The 46 month gap from 2021 to 2024 looks like the author moving to a new job since his commit email changed to a company address. The only commits after the gap are one day of CI cleanup in Jan 2025 so the project is still inactive.',
    reflection='''## Inactivity pattern
Most of the work on retina happened from 2014 to 2018. There were big spikes in 2017 and 2018. After 2019 it slowed way down and then it stopped for 46 months (March 2021 to December 2024). That is the longest gap of all my projects. It has 6 gaps in total.

## Was the gap easy or hard to read?
Kind of in the middle. None of the commit messages say "we are stopping". But there are some good clues. The README says the tool goes with a 2015 Journal of Vision paper so the main goal was already done. Also in the last commits before the gap the author Brian Cohn was using a fibronostics.com email instead of his usc.edu email. That makes me think he got a job outside the university. The last messages were "closing out juan-changes" and "tutorial test" which sound like he was wrapping things up. Issue #53 was opened in January 2021 right before the gap and it is still open.

## Likely reason for the gap and recovery
I think the author moved on to a new job and the project was already finished from a research point of view. It did not really recover. In January 2025 he made 5 commits all on the same day. They only change CI files add a repomix.yml file and remove tutorial.md. That is cleanup and not real new work. Nobody new joined and nothing happened after that.
'''),

 'caseywdunn_siphonophore_phylogeny_2017': dict(
    BeforeThemes='Documentation updates:Other',
    AfterThemes='Other:Documentation updates',
    RecentThemes='Documentation updates:Other',
    HypothesizedGapReason='The paper was done. This repo holds the analysis for Munro et al. 2018 in Molecular Phylogenetics and Evolution. The last commits before the gap are about the paper (supplementary figures and fixing typos and references and a graphical abstract and a README link to the MPE paper) and then uploading the transcriptome files in 2019. Once the paper and data were out there was nothing left to do.',
    HasRecovered='No',
    WhyRecovered='It did not recover. There are only 2 commits after the gap. One adds an SRA biosamples file (Feb 2023) and one adds a note that Nanomia bijuga Pacific is now called Nanomia septata (Oct 2024) after a 2024 taxonomy paper. Both just keep the record up to date.',
    WhoRecovered='Casey Dunn (the lab lead) made both later commits. He was already a contributor before the gap. No new people.',
    Currentstatus='Inactive',
    Notes='This is the code for one research paper so almost all the work happened in 2017 and 2018. The 42 month gap was easy to explain because the last commits were finishing the paper and sharing the data. The 2 commits after it only update data links and a species name so the project is done rather than abandoned.',
    reflection='''## Inactivity pattern
Almost all of the 437 commits happened in 2017 and 2018. After that there are only 3 commits total (2019 and 2023 and 2024). The longest gap is 42 months from August 2019 to January 2023. It has 3 gaps.

## Was the gap easy or hard to read?
Easy. The README says this repo is the analysis for one paper: Munro et al. 2018 in Molecular Phylogenetics and Evolution. The commits right before the gap are all about finishing that paper. They add supplementary figures and fix typos and references and make a graphical abstract and update the README to link to the paper. The very last ones upload the transcriptome data. So the project ended because the paper was published and the data was shared.

## Likely reason for the gap and recovery
The project was finished not abandoned. It did not really come back. Casey Dunn who runs the lab made the only 2 later commits. One adds an SRA biosamples file and the other adds a note that one species was renamed (Nanomia septata) because of a newer 2024 paper. Those are just small updates to keep the record correct. No new people joined.
'''),

 'giaf_hpipm': dict(
    BeforeThemes='Feature development:Documentation updates',
    AfterThemes='Bug fixes:Feature development',
    RecentThemes='Feature development:Bug fixes',
    HypothesizedGapReason='A short break by the main maintainer. giaf (Gianluca Frison) made most of the commits and the last ones before the gap were finishing a feature (objective computation and a MATLAB/Octave QCQP interface) and then updating the README in Nov 2022. That looks like a natural stopping point and the 5 months also cover the holidays. There is no sign of a problem in the messages.',
    HasRecovered='Yes',
    WhyRecovered='giaf came back in May 2023 and spent the next weeks fixing bugs and portability issues (32 bit alignment and header files and compiler warnings and timers). After that the project grew again from 2023 to 2025 with new features and more outside pull requests.',
    WhoRecovered='The same main maintainer (giaf) restarted it. After the gap new people also joined. 10 of the 15 authors after the gap are new (for example sandmaennchen).',
    Currentstatus='Active',
    Notes='hpipm was very busy from 2017 to 2020 then slowed down in 2021 and 2022 and picked up again from 2023 (U-shaped). The 5 month gap after Nov 2022 looks like the main developer taking a break after finishing a feature. He came back with a round of bug fixes and new contributors joined after that so the project is active again.',
    reflection='''## Inactivity pattern
hpipm was really busy from 2017 to 2020. It slowed down a lot in 2021 and 2022 and then picked back up from 2023 to 2025. That is why I called it U-shaped. The longest gap is 5 months from December 2022 to April 2023 and it has 2 gaps in total.

## Was the gap easy or hard to read?
Fairly easy but there is no direct explanation. Right before the gap giaf (the main developer) finished a feature for computing the objective value and added a MATLAB/Octave interface. His last two commits were README updates. That feels like a natural stopping point. The gap also covers the holidays. Nothing in the messages says anything went wrong.

## Likely reason for the gap and recovery
I think the main developer just took a break after finishing a chunk of work. When he came back in May 2023 the first commits were mostly bug fixes like fixing a 32 bit alignment issue and compiler warnings and header files. After that the project got more active again. The same main person restarted it but new people joined later too. 10 of the 15 people who committed after the gap had never committed before.
'''),

 'iic-jku_ddsim': dict(
    BeforeThemes='Feature development:Bug fixes',
    AfterThemes='Bug fixes:Feature development',
    RecentThemes='Automated bot contributions:Dependency updates',
    HypothesizedGapReason='N/A active Project',
    HasRecovered='N/A',
    WhyRecovered='N/A (no real gap. The only zero month was May 2020 right at the start of the project.)',
    WhoRecovered='N/A (the same two developers Stefan Hillmich and Lukas Burgholzer kept working and many new people and bots joined later)',
    Currentstatus='Active',
    Notes='ddsim has been steadily active since 2020 with 500 to 800 commits a year and no gaps of 3 months or more. The only empty month was May 2020 at the very start. Lately a lot of commits come from bots like renovate and dependabot that keep dependencies up to date.',
    reflection='''## Inactivity pattern
ddsim does not really have an inactivity gap. The only month with zero commits was May 2020 which was right at the start. Since then it has had 500 to 800 commits a year every year. It has 0 gaps of 3 months or more.

## Was the gap easy or hard to read?
There was not much to read. The one empty month happened while the two original developers (Stefan Hillmich and Lukas Burgholzer) were building the first simulators and benchmarks. They were back in June 2020 adding features and fixing builds. So this is just a normal pause early on.

## Activity pattern
This project is steady and healthy. 31 of the 33 people who committed after that first month were new so a lot of people joined over time. One thing I noticed is that the most recent commits are all from renovate[bot] updating pre-commit hooks and lock files. Bots like renovate and dependabot and pre-commit-ci made a big share of the commits. So part of the commit count comes from automated dependency updates and not from people writing code.
'''),

 'malbecc_lio': dict(
    BeforeThemes='Feature development:Bug fixes',
    AfterThemes='Other:Feature development',
    RecentThemes='Feature development:Other',
    HypothesizedGapReason='The main developer stopped and the code was later moved to a new repository. Before the gap almost all commits (422) were by one person (v01d) working on precision and force calculations and run scripts. That work ended in Sep 2011. The first commit after the gap (Jul 2013) is literally Commit inicial del proyecto LIO (initial commit of the LIO project) and it removes binaries and temporary files. So the project was restarted as a cleaned up new repo by a different group.',
    HasRecovered='Yes',
    WhyRecovered='A new team restarted it as the official LIO project in Jul 2013. They cleaned up the code and added a README and example runs and made a standalone version (liosolo) that runs without AMBER. It then grew a lot from 2014 to 2020 and the README lists papers from 2014 to 2018 that use it.',
    WhoRecovered='Mostly new people. 41 of the 42 authors after the gap are new (for example Manuel Ferreria and later fedepedron and gonzalodm). Only nanolebrero (Mariano Gonzalez Lebrero) was there before the gap and he stayed.',
    Currentstatus='Active',
    Notes='lio goes up and down a lot. The 21 month gap from 2011 to 2013 makes sense because the one main developer stopped and then a new team re-created the project with an initial commit in 2013 and it took off. It got quiet again after 2021 but there were still commits in Sep 2025 (on branches other than master) so it counts as active.',
    reflection='''## Inactivity pattern
lio is the oldest project I have (2007). It goes up and down a lot. It was small from 2007 to 2011 then stopped for 21 months (October 2011 to June 2013). It was very busy from 2014 to 2020 and then got quiet again after 2021 with a small pickup in 2025. It has 7 gaps.

## Was the gap easy or hard to read?
Pretty easy once you look at the messages (they are in Spanish). Before the gap almost every commit is from one person called v01d. He was working on number precision and force calculations and scripts to run tests. His last commits were in September 2011. The first commit after the gap in July 2013 says "Commit inicial del proyecto LIO" which means "initial commit of the LIO project". It also says they removed all the binaries and temporary files. So the code was moved into a new clean repo.

## Likely reason for the gap and recovery
I think v01d finished his part (maybe a thesis or a project) and stopped. Then in 2013 a group restarted it as the official LIO project. They added a README and example runs and a "liosolo" version that runs without AMBER. Almost everyone after the gap was new (41 of 42 authors). Only Mariano Gonzalez Lebrero (nanolebrero) was around before and after. The new team made it grow a lot.

## Extra note
GitHub's master branch shows the last commit in January 2024 but WoC has commits up to September 2025. Those newer commits must be on other branches. That is why I marked it Active.
'''),

 'neurodatawithoutborders_matnwb': dict(
    BeforeThemes='Documentation updates:Feature development',
    AfterThemes='Other:Feature development',
    RecentThemes='Documentation updates:Other',
    HypothesizedGapReason='N/A active Project',
    HasRecovered='N/A',
    WhyRecovered='N/A (the longest pause was only 2 months in Dec 2017 and Jan 2018 right at the start)',
    WhoRecovered='N/A (activity continued with Lawrence and a MathWorks developer Mark Cafaro who added the test suite in Feb 2018)',
    Currentstatus='Active',
    Notes='matnwb has been active the whole time and is growing with 1214 commits in 2025. The only pause was 2 months over the 2017 to 2018 holidays right after the first version and README were added. Recent work is mostly documentation and tutorial updates by ehennestad.',
    reflection='''## Inactivity pattern
matnwb never really went inactive. The longest pause was 2 months (December 2017 and January 2018) right at the very start. It has 0 gaps of 3 months or more. It keeps growing and had its biggest year in 2025 with 1214 commits.

## Was the gap easy or hard to read?
Easy. Right before the pause they had just finished the first version. The last commits added a license and a schema and a README. Then the pause covered the winter holidays. In February 2018 Mark Cafaro (with a mathworks.com email) added a whole test suite in one day. So it was just a short break after the first release.

## Activity pattern
This one is rising. A lot of new people joined over time (28 of the 30 authors after the pause were new). The most active person now is ehennestad and the latest commits are mostly updates to tutorials and docs. It is clearly an active and healthy project.
'''),

 'prjemian_spec2nexus': dict(
    BeforeThemes='Automated bot contributions:Dependency updates',
    AfterThemes='Release:Documentation updates',
    RecentThemes='Automated bot contributions:Release',
    HypothesizedGapReason='The project was in maintenance mode. Before the gap almost all commits were dependabot bumping GitHub Actions versions and the maintainer merging those PRs. There was no real feature work left so when the bot had nothing to update there were no commits. Pete Jemian is basically the only maintainer so the project depends on his free time.',
    HasRecovered='Partly',
    WhyRecovered='In May 2025 the maintainer came back to finish issue #303 (review packaging) which he opened in May 2023. He fixed the package build and updated the copyright and CI and set up PyPI Trusted Publisher and made release 2021.2.7. After that it went back to mostly dependabot updates.',
    WhoRecovered='Same person (Pete R Jemian) plus dependabot. No new people joined.',
    Currentstatus='Active',
    Notes='spec2nexus works in bursts with 9 gaps. The 12 month gap in 2024 and 2025 happened when the project was only getting dependabot updates. The maintainer came back in May 2025 to fix packaging (issue #303) and push out a release and then it went back to mostly bot updates.',
    reflection='''## Inactivity pattern
spec2nexus works in bursts. It gets a lot of commits all at once and then goes quiet. It has 9 gaps which is the second most. The longest one is 12 months from May 2024 to April 2025.

## Was the gap easy or hard to read?
Fairly easy. Before the gap almost every commit was dependabot bumping versions of GitHub Actions and Pete Jemian merging those. There was no new feature work. So the project was already in maintenance mode and when the bot stopped there was nothing else happening.

## Likely reason for the gap and recovery
The project is pretty much done and Pete Jemian is the only real maintainer so it depends on when he has time. In May 2025 he came back to work on issue #303 "review packaging" which he had opened back in May 2023. In one day he fixed the package build and updated the copyright and CI and set up PyPI Trusted Publisher and made a release (2021.2.7). After that it went back to mostly dependabot updates. So it recovered a little but only for a release. The same person did it and nobody new joined.
'''),

 'spirit-code_spirit': dict(
    BeforeThemes='Feature development:Other',
    AfterThemes='Other:Dependency updates',
    RecentThemes='Documentation updates:Feature development',
    HypothesizedGapReason='N/A active Project',
    HasRecovered='N/A',
    WhyRecovered='N/A (the only zero month was Jan 2023)',
    WhoRecovered='N/A (G. P. Mueller kept working and puerling became the main contributor later)',
    Currentstatus='Active',
    Notes='spirit never stopped. The only empty month was Jan 2023 and the work right after was CI and test cleanup. GitHub shows the master branch last updated in 2023 but WoC has commits up to Oct 2025 so the active work is on other branches like develop.',
    reflection='''## Inactivity pattern
spirit never really went inactive. The only month with zero commits was January 2023 and it has 0 gaps of 3 months or more. It was busiest in 2016 and 2017 and got slower until 2022 and 2023. Then it went back up in 2024 which is why I called it U-shaped.

## Was the gap easy or hard to read?
Easy. In December 2022 Moritz Sallermann was working on solver changes and fixed a bug. After the one empty month G. P. Mueller came back in February 2023 and spent time updating CI and the test library and cleaning up tests. So it was just a short holiday pause.

## Activity pattern and current status
The project kept going and a new main contributor (puerling) showed up after 2023 with over 600 commits. One confusing thing is that GitHub says the last commit was in March 2023. That date is only for the master branch. WoC has 3001 commits but GitHub master only shows about 1900. WoC also has commits up to October 2025 (docs and logging work). So the real work happens on other branches like develop and the project is active.
'''),

 'stanfordnlp_glove': dict(
    BeforeThemes='Bug fixes:Feature development',
    AfterThemes='Bug fixes:Documentation updates',
    RecentThemes='Documentation updates:Bug fixes',
    HypothesizedGapReason='Nobody at Stanford was actively working on it. The commits before the gap were small fixes sent in by outside people (a memory overflow fix and a context vector output option and a build fix). GloVe is a finished research tool from 2014 so the original team had moved on and the repo only changed when someone sent a pull request. Several commits are just GitHub test merges of pull requests that were never merged (for example Erin George).',
    HasRecovered='Yes',
    WhyRecovered='Two things. First NumPy 2.0 (released 2024) removed np.Inf which broke the evaluation script so people sent fixes in Nov 2024 and Jul 2025. Second Stanford trained new 2024 vectors and in Jul 2025 rlyCarlson (who has a Stanford email) updated the docs and linked the new vectors. The README now has a NEW 2024 VECTORS section and links a report (arXiv 2507.18103).',
    WhoRecovered='A mix. 2 of the 4 people after the gap were new (leo9827 and Jesper Olsen who sent fixes). rlyCarlson (Stanford email) who updated the docs had committed before. The original 2014 authors did not come back.',
    Currentstatus='Active',
    Notes='GloVe is famous (7230 stars) but gets very few commits and has 11 gaps. The 13 month gap in 2023 and 2024 happened because nobody was maintaining it and changes only came from outside pull requests. It came back in 2024 and 2025 because NumPy 2.0 broke a script and because Stanford released new 2024 vectors.',
    reflection='''## Inactivity pattern
GloVe has the most stars of all my projects (7230) but very few commits (263). It has 11 gaps which is the most. The longest gap is 13 months from October 2023 to October 2024. It got a small amount of work regularly until 2020 and after that only a few commits a year.

## Was the gap easy or hard to read?
Medium. No message says why it stopped. But the commits before the gap were all small fixes sent in by outside people like a memory overflow fix for huge machines and an option to output context vectors. GloVe came out in 2014 so the original Stanford team had moved on. The repo only changed when someone sent a pull request. Also some of the commits in WoC are GitHub's test merges for pull requests that were never actually merged (like the ones from Erin George). That makes the project look a bit more active than it really was.

## Likely reason for the gap and recovery
I think it went quiet because nobody was in charge of it. It came back for two reasons. First NumPy 2.0 came out in 2024 and removed np.Inf which broke the evaluation script. People sent fixes for that in November 2024 and July 2025. Second Stanford trained new 2024 GloVe vectors. In July 2025 rlyCarlson (who has a Stanford email) updated the docs and linked the new vectors. The README now has a "NEW 2024 VECTORS" section. Half of the people after the gap were new (leo9827 and Jesper Olsen) and the original authors did not come back.
'''),
}

## Updates smaciasa_project_stats.csv  Reflections

In [ ]:
new_cols = ['BeforeThemes', 'AfterThemes', 'HypothesizedGapReason', 'HasRecovered',
            'WhyRecovered', 'WhoRecovered', 'Currentstatus', 'RecentThemes', 'Notes']
for c in new_cols:
    stats[c] = stats['Project'].map(lambda p: analysis[p][c])
    assert not stats[c].str.contains(';').any(), c   # keep the CSV semicolon-safe
stats.to_csv(f'{netid}_project_stats.csv', sep=';', index=False)

for p in stats['Project']:
    with open(f'{netid}_reflection_{p}.md', 'w') as f:
        f.write(f'# Reflection: {p}\n\n' + analysis[p]['reflection'].strip() + '\n\n## Summary\n' + analysis[p]['Notes'] + '\n')

print(list(stats.columns))
stats[['Project', 'BeforeThemes', 'AfterThemes', 'HasRecovered', 'Currentstatus', 'RecentThemes']]

['Project', 'ncommits', 'nauthors', 'from', 'to', 'nstars', 'nforks', 'lastGHCommitDate', 'LongestGapStart', 'LongestGapEnd', 'LongestGapLength', 'NumCommitsAfterLastGap', 'ActivityPattern', 'NumberOfGapsInTimeline', 'BeforeThemes', 'AfterThemes', 'HypothesizedGapReason', 'HasRecovered', 'WhyRecovered', 'WhoRecovered', 'Currentstatus', 'RecentThemes', 'Notes']


,Project,BeforeThemes,AfterThemes,HasRecovered,Currentstatus,RecentThemes
0,antonyuryev_elsevierapi,Other:Feature development,Other:Feature development,Yes,Active,Other:Feature development
1,bc_retina,Other:Feature development,Other:Documentation updates,No,Inactive,Other:Documentation updates
2,caseywdunn_siphonophore_phylogeny_2017,Documentation updates:Other,Other:Documentation updates,No,Inactive,Documentation updates:Other
3,giaf_hpipm,Feature development:Documentation updates,Bug fixes:Feature development,Yes,Active,Feature development:Bug fixes
4,iic-jku_ddsim,Feature development:Bug fixes,Bug fixes:Feature development,N/A,Active,Automated bot contributions:Dependency updates
5,malbecc_lio,Feature development:Bug fixes,Other:Feature development,Yes,Active,Feature development:Other
6,neurodatawithoutborders_matnwb,Documentation updates:Feature development,Other:Feature development,N/A,Active,Documentation updates:Other
7,prjemian_spec2nexus,Automated bot contributions:Dependency updates,Release:Documentation updates,Partly,Active,Automated bot contributions:Release
8,spirit-code_spirit,Feature development:Other,Other:Dependency updates,N/A,Active,Documentation updates:Feature development
9,stanfordnlp_glove,Bug fixes:Feature development,Bug fixes:Documentation updates,Yes,Active,Documentation updates:Bug fixes
